In [19]:
import gradio as gr
import pandas as pd
import json
import tempfile
import os
from openai import OpenAI

In [23]:
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-...")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"

# Model via OpenRouter — Claude Sonnet is a great balance of speed & quality.
# Other options: "anthropic/claude-opus-4", "anthropic/claude-haiku-4-5"
MODEL = "anthropic/claude-sonnet-4-5"

print(f"Using model : {MODEL}")
print(f"API key set : {'Yes' if not OPENROUTER_API_KEY.endswith('...') else 'No — please paste your key above'}") 

Using model : anthropic/claude-sonnet-4-5
API key set : Yes


In [36]:
def generate_synthetic_data(model_name, topic, num_rows, columns_desc):
    if not topic:
        return pd.DataFrame(), "Please enter a topic first."
    
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    # Smart Schema Logic
    schema_info = columns_desc if columns_desc.strip() else "Design 5-7 professional, relevant columns for this topic."
    
    system_prompt = "You are a data architect. Output ONLY a JSON array of objects. No intro text."
    user_prompt = f"Topic: {topic}. Rows: {num_rows}. Schema: {schema_info}"

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={ "type": "json_object" },
            timeout=45.0 
        )
        
        content = response.choices[0].message.content
        data = json.loads(content)
        
        # Handle various JSON nesting styles from different LLMs
        if isinstance(data, dict):
            for v in data.values():
                if isinstance(v, list):
                    data = v
                    break
            
        df = pd.DataFrame(data)
        return df, f"Successfully generated {len(df)} rows. Use the icon in the top right of the table to copy data."

    except Exception as e:
        return pd.DataFrame(), f"Error: {str(e)}"

In [34]:
def reset_ui():
    return "openai/gpt-4o-mini", "", 10, "", pd.DataFrame(), "Ready"

In [37]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 AI Data Architect")
    gr.Markdown("Enter a topic to generate structured data. Copy the results using the table icon.")
    
    with gr.Row():
        with gr.Column(scale=1):
            model_drop = gr.Dropdown(
                choices=["openai/gpt-4o-mini", "anthropic/claude-3.5-sonnet", "google/gemini-flash-1.5"], 
                value="openai/gpt-4o-mini", label="Select AI Model"
            )
            topic_txt = gr.Textbox(label="Data Topic", placeholder="e.g. E-commerce Customer List")
            rows_sldr = gr.Slider(1, 50, 10, step=1, label="Number of Rows")
            schema_txt = gr.Textbox(label="Columns (Optional)", placeholder="Leave blank for AI to guess")
            
            with gr.Row():
                gen_btn = gr.Button("🚀 Generate", variant="primary")
                clear_btn = gr.Button("🗑️ Reset", variant="stop")
            
            status_out = gr.Textbox(label="Status Log", value="Ready", interactive=False)

        with gr.Column(scale=2):
            # show_copy_button allows one-click copying to Excel/Sheets
            output_df = gr.DataFrame(
                label="Data Preview", 
                interactive=False, 
                show_copy_button=True
            )

    gen_btn.click(
        fn=generate_synthetic_data,
        inputs=[model_drop, topic_txt, rows_sldr, schema_txt],
        outputs=[output_df, status_out]
    )

    clear_btn.click(
        fn=reset_ui,
        inputs=[],
        outputs=[model_drop, topic_txt, rows_sldr, schema_txt, output_df, status_out]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
